In [12]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.models import load_model


In [13]:
word_index = imdb.get_word_index()
reverse_word_index = {value:key for key, value in word_index.items()} 

In [14]:
from tensorflow.keras import Input

model = Sequential([
    Input(shape=(500,)),
    Embedding(10000,128),
    SimpleRNN(128),
    Dense(1,activation='sigmoid')
])

In [15]:
model.save("simple_rnn_imdb.h5")

In [16]:
from tensorflow.keras.models import load_model

model = load_model("simple_rnn_imdb.h5")
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.get_weights()

[array([[-0.02709251, -0.0346378 , -0.0306847 , ...,  0.03931979,
          0.02193565, -0.03330801],
        [ 0.02143749,  0.00018458,  0.04569909, ...,  0.00974213,
          0.03686173, -0.02960104],
        [-0.00358867, -0.03160948, -0.03586304, ..., -0.01228224,
          0.04183063, -0.01551409],
        ...,
        [ 0.01281604, -0.00948819, -0.04242413, ...,  0.00550792,
          0.02160661, -0.0051612 ],
        [-0.03359659,  0.04084314,  0.03842762, ...,  0.04400029,
         -0.02930707, -0.0330053 ],
        [ 0.03890482,  0.00015898,  0.04436084, ...,  0.04192784,
          0.03134974, -0.02731254]], shape=(10000, 128), dtype=float32),
 array([[ 0.08565694, -0.04912681, -0.05072636, ..., -0.00883149,
          0.0332873 ,  0.04989098],
        [ 0.09403221, -0.05799578, -0.05456754, ..., -0.14788544,
          0.1368909 , -0.12499702],
        [-0.05181845,  0.035731  ,  0.10661225, ...,  0.10748975,
          0.07435147,  0.12134688],
        ...,
        [-0.0116231

In [18]:
def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

In [19]:
def preprocess_text(text):
    # Tokenize the text into words
    words = text.lower().split()
    
    # Convert words to their corresponding indices in the word index
    encoded_review = [word_index.get(word, 2) + 3 for word in words]  # 2 is for unknown words
    
    # Pad the sequence to ensure it has a length of 500
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    
    return padded_review

In [20]:
##prediction func
def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input)
    sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"
    return sentiment, prediction[0][0]


In [21]:
##user input and pred
example_review = "This movie was fantastic! I loved it."
sentiment, score = predict_sentiment(example_review)
print(f"Review: {example_review}\nPredicted Sentiment: {sentiment} (Score: {score})")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
Review: This movie was fantastic! I loved it.
Predicted Sentiment: Negative (Score: 0.4832504987716675)


In [22]:
from tensorflow.keras.models import load_model

model = load_model("simple_rnn_imdb.h5")